# Telco Customer Churn Analysis with scikit-learn

This notebook analyzes customer behavior and churn risk using the Kaggle Telco Customer Churn dataset: https://www.kaggle.com/datasets/blastchar/telco-customer-churn

It covers data cleaning, preprocessing, feature engineering, model selection (Logistic Regression, Decision Tree, Random Forest, XGBoost), evaluation, and SHAP explainability.

## 1) Setup

In [ ]:
# If needed, uncomment:
# %pip install -q pandas numpy scikit-learn xgboost shap matplotlib seaborn

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

import shap

pd.set_option('display.max_columns', 100)
sns.set_theme(style='whitegrid')

## 2) Load data

In [ ]:
# Put WA_Fn-UseC_-Telco-Customer-Churn.csv in ./data or update this path.
data_path = Path('data/WA_Fn-UseC_-Telco-Customer-Churn.csv')
if not data_path.exists():
    raise FileNotFoundError(f'Expected dataset at: {data_path.resolve()}')

df = pd.read_csv(data_path)
print('Shape:', df.shape)
df.head()

## 3) Data cleaning and preprocessing

In [ ]:
df = df.copy()

# Clean known numeric column encoded as string in this dataset
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# Binary encode target
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

# Feature engineering for customer behavior / payment-risk proxies
tenure_nonzero = df['tenure'].replace(0, np.nan)
df['AvgMonthlyCharge'] = df['TotalCharges'] / tenure_nonzero
df['HasFiberOptic'] = (df['InternetService'] == 'Fiber optic').astype(int)
df['HasElectronicCheck'] = (df['PaymentMethod'] == 'Electronic check').astype(int)

X = df.drop(columns=['Churn', 'customerID'])
y = df['Churn']

num_cols = X.select_dtypes(include=['number']).columns.tolist()
cat_cols = X.select_dtypes(exclude=['number']).columns.tolist()

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, num_cols),
        ('cat', categorical_transformer, cat_cols)
    ]
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print('Train shape:', X_train.shape)
print('Test shape:', X_test.shape)

## 4) Quick behavior and churn exploration

In [ ]:
churn_by_contract = df.groupby('Contract', as_index=False)['Churn'].mean().sort_values('Churn', ascending=False)
churn_by_payment = df.groupby('PaymentMethod', as_index=False)['Churn'].mean().sort_values('Churn', ascending=False)

display(churn_by_contract)
display(churn_by_payment)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.barplot(data=churn_by_contract, x='Contract', y='Churn', ax=axes[0])
axes[0].set_title('Churn Rate by Contract Type')
axes[0].tick_params(axis='x', rotation=20)

sns.barplot(data=churn_by_payment, x='PaymentMethod', y='Churn', ax=axes[1])
axes[1].set_title('Churn Rate by Payment Method (Credit/Payment Behavior Proxy)')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

## 5) Model training and evaluation

In [ ]:
models = {
    'LogisticRegression': LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42),
    'DecisionTree': DecisionTreeClassifier(max_depth=6, random_state=42),
    'RandomForest': RandomForestClassifier(n_estimators=300, random_state=42, class_weight='balanced', n_jobs=-1),
    'XGBoost': XGBClassifier(
        n_estimators=400,
        learning_rate=0.05,
        max_depth=4,
        subsample=0.9,
        colsample_bytree=0.9,
        eval_metric='logloss',
        random_state=42
    )
}

results = []
fitted_pipelines = {}

for name, model in models.items():
    pipe = Pipeline(steps=[('preprocessor', preprocessor), ('model', model)])
    pipe.fit(X_train, y_train)

    y_pred = pipe.predict(X_test)
    y_proba = pipe.predict_proba(X_test)[:, 1]

    results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1': f1_score(y_test, y_pred),
        'ROC_AUC': roc_auc_score(y_test, y_proba),
    })
    fitted_pipelines[name] = pipe

results_df = pd.DataFrame(results).sort_values('ROC_AUC', ascending=False).reset_index(drop=True)
display(results_df)

best_model_name = results_df.loc[0, 'Model']
best_pipeline = fitted_pipelines[best_model_name]
print('Best model by ROC-AUC:', best_model_name)

## 6) Explainability with SHAP on the best model

In [ ]:
X_sample = X_test.sample(min(1000, len(X_test)), random_state=42)
X_sample_t = best_pipeline.named_steps['preprocessor'].transform(X_sample)
feature_names = best_pipeline.named_steps['preprocessor'].get_feature_names_out()

best_estimator = best_pipeline.named_steps['model']

if best_model_name == 'LogisticRegression':
    explainer = shap.LinearExplainer(best_estimator, X_sample_t)
    shap_values = explainer.shap_values(X_sample_t)
    shap.summary_plot(shap_values, features=X_sample_t, feature_names=feature_names, show=False)
else:
    explainer = shap.TreeExplainer(best_estimator)
    shap_values = explainer.shap_values(X_sample_t)
    if isinstance(shap_values, list):
        shap_values = shap_values[1] if len(shap_values) > 1 else shap_values[0]
    if hasattr(shap_values, 'ndim') and shap_values.ndim == 3:
        shap_values = shap_values[:, :, 1]
    shap.summary_plot(shap_values, features=X_sample_t, feature_names=feature_names, show=False)

plt.title(f'SHAP Summary - {best_model_name}')
plt.tight_layout()
plt.show()

## 7) Business interpretation

Use the model ranking and SHAP plot to identify top churn drivers. In this dataset, common churn indicators are typically contract type, tenure, monthly charges, internet service profile, and payment method behavior.